# Project 1: Deep Research Agent

**30 min project**

Build an agent that researches a topic, uses tools, and returns a structured research brief.

**Concepts:** `Agent`, `Runner`, `WebSearchTool`, `@function_tool`, `output_type`

In [ ]:
!pip install -q openai-agents python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

## 1. Basic Research Agent

In [ ]:
from agents import Agent, Runner

research_agent = Agent(
    name="Research Agent",
    instructions="You are a concise research assistant. Answer with key facts, tradeoffs, and practical implications.",
    model="gpt-4o-mini",
)

result = Runner.run_sync(
    research_agent,
    "What are the main benefits of using multi-agent systems for software engineering teams?",
)

print(result.final_output)
print("\nAnswered by:", result.last_agent.name)

## 2. Add Search + Research Tools

In [ ]:
import re
from agents import WebSearchTool, function_tool


@function_tool
def extract_key_terms(topic: str) -> list[str]:
    """Extract likely research keywords from a topic."""
    words = re.findall(r"[A-Za-z][A-Za-z0-9-]+", topic.lower())
    stopwords = {"the", "and", "for", "with", "from", "into", "what", "why", "how", "are", "is", "to", "of", "in"}
    return [w for w in words if w not in stopwords][:8]


@function_tool
def score_source_coverage(source_count: int) -> str:
    """Score how well-supported a research answer is based on source count."""
    if source_count >= 5:
        return "strong coverage"
    if source_count >= 3:
        return "moderate coverage"
    return "light coverage"


tooled_research_agent = Agent(
    name="Tool-Using Research Agent",
    instructions=(
        "Research the user's topic. Use web search for current information. "
        "Use the helper tools to identify key terms and reason about source coverage. "
        "Keep the answer concise and practical."
    ),
    model="gpt-4o-mini",
    tools=[WebSearchTool(), extract_key_terms, score_source_coverage],
)

result = Runner.run_sync(
    tooled_research_agent,
    "Research: What are the best use cases for MCP servers in AI engineering workflows?",
)

print(result.final_output)

## 3. Structured Research Brief

In [ ]:
from pydantic import BaseModel, Field


class Finding(BaseModel):
    claim: str
    evidence: str
    implication: str


class ResearchBrief(BaseModel):
    topic: str
    key_findings: list[Finding]
    sources: list[str]
    confidence_score: int = Field(description="Confidence from 1 to 10")
    next_steps: list[str]


structured_research_agent = Agent(
    name="Structured Research Agent",
    instructions=(
        "Create a structured research brief. Use web search for current information. "
        "Every finding must include a claim, evidence, and practical implication. "
        "Include source URLs or source names when available."
    ),
    model="gpt-4o-mini",
    tools=[WebSearchTool(), extract_key_terms, score_source_coverage],
    output_type=ResearchBrief,
)

result = Runner.run_sync(
    structured_research_agent,
    "Research current best practices for evaluating AI agents in production.",
)

brief = result.final_output
print("Topic:", brief.topic)
print("Confidence:", brief.confidence_score)
print("\nFindings:")
for finding in brief.key_findings:
    print("-", finding.claim)
    print("  Evidence:", finding.evidence)
    print("  Implication:", finding.implication)
print("\nSources:", brief.sources)

## 4. Typed Output Is Usable Data

In [ ]:
print(type(brief))
print("\nJSON:")
print(brief.model_dump_json(indent=2))

high_confidence = brief.confidence_score >= 7
print("\nReady to share with team:", high_confidence)

## 5. Try Another Research Query

In [ ]:
query = "Research whether small engineering teams should use multi-agent workflows or single-agent workflows."

result = Runner.run_sync(structured_research_agent, query)
brief = result.final_output

print(f"{brief.topic} ({brief.confidence_score}/10)")
for i, finding in enumerate(brief.key_findings, 1):
    print(f"\n{i}. {finding.claim}")
    print(f"   {finding.implication}")